In [27]:
import numpy as np
import pandas as pd
import json
import datetime
import pickle

In [2]:
# load in psiturk data
rm1df = pd.read_json('../../data/db/exported/room1-2.11.19.json', convert_dates=['beginhit','endhit'])
rm2df = pd.read_json('../../data/db/exported/room2-2.11.19.json', convert_dates=['beginhit','endhit'])

# drop runs that didn't finish
rm1df = rm1df[rm1df.status != 1]
rm2df = rm2df[rm2df.status != 1]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)
rm2df = rm2df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)

# format datastring as dict 
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# add test room column
rm1df['testroom'] = 1
rm2df['testroom'] = 2

# remove test runs for each room
rm1df = rm1df.loc[1:]
rm2df = rm2df.loc[1:]

# concatenate dataframes, order by start time
expdf = pd.concat([rm1df, rm2df], ignore_index=True).sort_values('beginhit').reset_index(drop=True)

# drop subjects 
expdf = expdf.loc[expdf.index != 7].reset_index(drop=True) # experiment error
expdf = expdf.loc[expdf.index != 56].reset_index(drop=True) # no-show for part 2
expdf = expdf.loc[expdf.index != 62].reset_index(drop=True) # no-show for part 2

In [3]:
# load pre/post questionnaire responses
preqdf = pd.read_csv('../../data/google-form-data/Pre-experiment Questionnaire.csv', parse_dates=[0])
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
postqdf = pd.read_csv('../../data/google-form-data/Post-experiment questionnaire.csv', parse_dates=[0])
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})

# exclude test runs
preqdf = preqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)
postqdf = postqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)

In [4]:
# convert from datetime to POSIX time **SERIES.APPLY(DT.DT.TIMESTAMP).MULTIPLY(1000) DOES NOT WORK**
# add 3 hours to convert from UTC to ET
# subtract 1 hours to EST times to account for datetime handling of DST
newpretimestamp = pd.Series([0]*len(preqdf['preqtime']))
for ix, val in enumerate(newpretimestamp):
    newpretimestamp[ix] = preqdf['preqtime'][ix].timestamp()*1000
    if ix <= 92:
        newpretimestamp[ix] += 3.6e6

preqdf['preqtime'] = newpretimestamp

newposttimestamp = pd.Series([0]*len(postqdf['postqtime']))
for ix, val in enumerate(newposttimestamp):
    newposttimestamp[ix] = postqdf['postqtime'][ix].timestamp()*1000
    if ix <= 92:
        newposttimestamp[ix] += 3.6e6

postqdf['postqtime'] = newposttimestamp

In [5]:
# remove dropped subjects from google form
dropids = ['MD-102218-B-04','MD-020119-A-01','MD-102318-A-01','MD-101318-A-05','MD-1011318-A-05','MD-020119-B-01',
           'MD-101218-B-04']

for dropid in dropids:
    preqdf = preqdf[preqdf['Subject ID'] != dropid]
    postqdf = postqdf[postqdf['Subject ID'] != dropid]
        
preqdf.reset_index(drop=True, inplace=True)
postqdf.reset_index(drop=True, inplace=True)

In [6]:
# fix mistaken day/repeat ID assignments

preqdf.at[76,'Subject ID'] = 'MD-102018-A-03'
postqdf.at[76,'Subject ID'] = 'MD-102018-A-03'

preqdf.at[77,'Subject ID'] = 'MD-102218-A-06'
postqdf.at[77,'Subject ID'] = 'MD-102218-A-06'

preqdf.at[85,'Subject ID'] = 'MD-102218-B-06'
postqdf.at[85,'Subject ID'] = 'MD-102218-B-06'

preqdf.at[89,'Subject ID'] = 'MD-013119-A-02'
postqdf.at[90,'Subject ID'] = 'MD-013119-A-02'

### for mapping between Google Forms with experiment IDs and SQLite databases with PsiTurk IDs

In [7]:
# for ix, row in expdf.iterrows():
#     print(datetime.datetime.fromtimestamp(row['datastring']['data'][0]['dateTime']/1000))
#     print(datetime.datetime.fromtimestamp(preqdf.loc[ix,'preqtime']/1000))
#     #print(preqdf.loc[ix,'preqtime'] - datetime.timedelta(hours=3))
#     print(ix)
#     print('_______')

In [8]:
# add empty columns from pre/postquestionnaires to expdf
newcols = pd.unique(np.concatenate([i.columns.values for i in [preqdf,postqdf]]))
expdf = pd.concat([expdf, pd.DataFrame(columns=newcols)], sort=False)

# separate session 1 and 2 postquestionnaires
ses1postqdf = postqdf.drop_duplicates('Subject ID')
tempdf = postqdf.copy(deep=True)
rowsin1 = [ix for ix, row in ses1postqdf.iterrows()]

for ix, row in tempdf.iterrows():
    if ix in rowsin1:
        tempdf.drop(ix, inplace=True)
ses2postqdf = tempdf

ses1postqdf.reset_index(drop=True, inplace=True)
ses2postqdf.reset_index(drop=True, inplace=True)

In [9]:
# add prequestionnaire responses to corresponding subject
for ix, row in expdf.iterrows():
    for col in preqdf.columns:
        expdf.at[ix,col] = preqdf.at[ix,col]

# add session 1 postquestionnaire to first occurence of each subject
for ix, row in expdf.drop_duplicates('Subject ID').iterrows():
    expdf.loc[ix,'postqtime':] = (ses1postqdf.loc[ses1postqdf['Subject ID'] == row['Subject ID']]).drop(columns='Subject ID').values[0]

# add session 2 postquestionnaire to rest
for ix, row in expdf.iterrows():
    if np.isnan(row['postqtime']):
        expdf.loc[ix,'postqtime':] = (ses2postqdf.loc[ses2postqdf['Subject ID'] == row['Subject ID']]).drop(columns='Subject ID').values[0]

In [10]:
expdf

,uniqueid,datastring,beginhit,endhit,hitid,status,testroom,preqtime,Subject ID,"Outside of this study, have you ever watched an episode of either of the TV shows ""Atlanta"" or ""Arrested Development?""",...,What is/was your major?,How many hours of sleep did you get last night?,How many cups of coffee have you had today?,How alert are you feeling?,postqtime,How engaging did you find the episode?,How easy/difficult was it to follow the episode?,How well do you feel you recalled the events of the episode?,How well do you feel you learned the characters' names over the course of the episode?,How tired do you feel?
0,debugIEH2T:debugDLVLJ,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 18:14:31.394716,2018-10-12 19:12:06.999050,debug7rmxU,3.0,1.0,1539368143000,MD-101218-A-01,I've never watched either one,...,undeclared,7,1,A little alert,1539371520000,Very engaging,Somewhat easy,Very well,Very well,A little tired
1,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 19:17:03.503575,2018-10-12 20:10:44.883411,debugTkKFp,3.0,1.0,1539371933000,MD-101218-B-01,I've never watched either one,...,undeclared,5,0,A little sluggish,1539374995000,A little engaging,Somewhat easy,Very well,Somewhat well,A little tired
2,debugYQfMB:debugxg7il,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 19:27:41.895057,2018-10-12 20:19:45.738723,debugnwalU,3.0,2.0,1539372538000,MD-101218-A-02,I've never watched either one,...,undeclared,7,2,A little alert,1539375570000,Very engaging,Somewhat easy,Somewhat well,Somewhat well,A little tired
3,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 20:22:07.319435,2018-10-12 21:09:46.566325,debugonOYk,3.0,1.0,1539375802000,MD-101218-B-02,I've never watched either one,...,neuroscience,9,0,A little alert,1539378568000,Very engaging,Very easy,Very well,Somewhat well,A little alert
4,debug92cgv:debugvdAIT,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 20:30:14.351756,2018-10-12 21:19:34.104523,debugxHkRX,3.0,2.0,1539376286000,MD-101218-A-03,I've never watched either one,...,Sociology,7,0,A little alert,1539379114000,Very engaging,Somewhat easy,Very well,Somewhat well,Very alert
5,debugGaDml:debugFTHoY,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 21:28:01.305584,2018-10-12 22:16:31.057617,debugXHY6O,3.0,1.0,1539379748000,MD-101218-B-03,I've never watched either one,...,undeclared,8,0,A little alert,1539382561000,A little engaging,Somewhat easy,Neutral,Neutral,Neutral
6,debugmFRWe:debug4RZkW,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 21:30:34.220872,2018-10-12 22:23:10.006172,debugfYl0n,3.0,2.0,1539379944000,MD-101218-A-04,I've never watched either one,...,Undeclared (but planning on being a psychology...,8,1,Very sluggish,1539382935000,Very engaging,Neutral,Somewhat well,Not very well,A little tired
7,debugokLIG:debugalG88,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:28:34.378931,2018-10-13 19:15:12.977411,debugslG65,3.0,1.0,1539455393000,MD-101318-A-01,I've never watched either one,...,undeclared,8,0,A little alert,1539458108000,A little engaging,Neutral,Neutral,Neutral,Neutral
8,debug3dOrm:debugAjPUS,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:33:46.272794,2018-10-13 19:31:06.315471,debugRcsF5,3.0,2.0,1539455701000,MD-101318-B-01,I've never watched either one,...,Undeclared,7,2,Neutral,1539459059000,Very engaging,Somewhat easy,Somewhat well,Somewhat well,Very alert
9,debugdhnfF:debug6fW93,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 19:35:43.889748,2018-10-13 20:32:37.663906,debugszpCm,3.0,1.0,1539459418000,MD-101318-B-02,I've never watched either one,...,Psychology,6,1,Neutral,1539462754000,Very engaging,Somewhat easy,Somewhat well,Not very well,A little tired


In [11]:
id_maps = {}

for ix, row in expdf.iterrows():
    if row['Subject ID'] in id_maps:
        id_maps[row['Subject ID']]['session 2'] = row['uniqueid']
    else:
        id_maps[row['Subject ID']] = {'session 1' : row['uniqueid'], 'session 2' : None}

In [12]:
id_maps

{'MD-013119-A-01': {'session 1': 'debughKp9W:debug1QFCO',
  'session 2': 'debugSeRNG:debug5pTo9'},
 'MD-013119-A-02': {'session 1': 'debug5ytRS:debug22nvn',
  'session 2': 'debug3Z74G:debug6CQ6D'},
 'MD-013119-B-01': {'session 1': 'debuguhHBa:debug1HOCZ',
  'session 2': 'debug0GSk9:debugttOEY'},
 'MD-020119-A-02': {'session 1': 'debugfGFHT:debugLCOgS',
  'session 2': 'debug9ngWi:debugpMjGl'},
 'MD-020119-A-03': {'session 1': 'debugRQQWb:debugwDi2H',
  'session 2': 'debugWRfIG:debugJWnAi'},
 'MD-020119-A-04': {'session 1': 'debugo1MpX:debugfpXYH',
  'session 2': 'debugeyj0U:debugsaXvf'},
 'MD-020119-B-02': {'session 1': 'debugAU1cu:debugzdGmZ',
  'session 2': 'debug27k0b:debugOGsyi'},
 'MD-020119-B-03': {'session 1': 'debugxQ7lq:debugS1UNb',
  'session 2': 'debug41SJm:debugJW4AU'},
 'MD-020719-B-01': {'session 1': 'debugwHDQh:debugDtNml', 'session 2': None},
 'MD-020819-B-01': {'session 1': 'debug3RFCM:debuggg1Gs', 'session 2': None},
 'MD-101218-A-01': {'session 1': 'debugIEH2T:debugDL

In [29]:
with open('../../data/pickles/expdf.p', 'wb') as file:
    pickle.dump(expdf, file)

In [30]:
with open('../../data/pickles/id_maps.p', 'wb') as file:
    pickle.dump(id_maps, file)